In [2]:
import sys
import os
import warnings
import json
import pickle
import joblib
from datetime import datetime
import glob

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Optuna for hyperparameter optimization
try:
    import optuna
    from optuna.samplers import TPESampler
    from optuna.pruners import MedianPruner
except ImportError:
    print("❌ Optuna not installed. Run: pip install optuna")

# MLflow
try:
    import mlflow
    import mlflow.sklearn
except ImportError:
    print("❌ MLflow not installed. Run: pip install mlflow")

# Model libraries
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, Lasso
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

# Suppress warnings
warnings.filterwarnings('ignore')

c:\Users\touhi\Desktop\ML Projects\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Configuration
CONFIG = {
    'data': {
        'train_path': '../data/splits/splits/X_train.csv',
        'val_path': '../data/splits/splits/X_val.csv',
        'test_path': '../data/splits/splits/X_test.csv',
        'y_train_low': '../data/splits/splits/y_train_low.csv',
        'y_train_mid': '../data/splits/splits/y_train_mid.csv',
        'y_val_low': '../data/splits/splits/y_val_low.csv',
        'y_val_mid': '../data/splits/splits/y_val_mid.csv'
    },
    'models': {
        'checkpoint_dir': '../models/checkpoints/',
        'tuned_dir': '../models/tuned/',
        'mlflow_dir': '../models/mlflow/'
    },
    'tuning': {
        'n_trials': 100,
        'cv_folds': 5,
        'random_state': 42,
        'timeout': 3600  # 1 hour max
    }
}

# Create directories
os.makedirs(CONFIG['models']['tuned_dir'], exist_ok=True)
os.makedirs(CONFIG['models']['mlflow_dir'], exist_ok=True)

print("✅ Configuration loaded!")
print(f"📁 Tuned models directory: {CONFIG['models']['tuned_dir']}")
print(f"📁 MLflow directory: {CONFIG['models']['mlflow_dir']}")

✅ Configuration loaded!
📁 Tuned models directory: ../models/tuned/
📁 MLflow directory: ../models/mlflow/


In [5]:
# Load Data and Models
def load_latest_checkpoint(checkpoint_dir, target_group):
    
    print(f"\n📂 Loading checkpoints for {target_group.upper()} income...")
    
    # Find the latest checkpoint files
    model_files = glob.glob(f"{checkpoint_dir}/model_{target_group}_*.pkl")
    result_files = glob.glob(f"{checkpoint_dir}/results_{target_group}_*.csv")
    
    if not model_files or not result_files:
        print(f"⚠️ No checkpoints found for {target_group.upper()} income")
        return None
    
    # Get the latest files (by creation time)
    latest_model = max(model_files, key=os.path.getctime)
    latest_results = max(result_files, key=os.path.getctime)
    
    print(f"  ✅ Found model: {os.path.basename(latest_model)}")
    print(f"  ✅ Found results: {os.path.basename(latest_results)}")
    
    # Load results
    results_df = pd.read_csv(latest_results)
    
    # Load models
    models = {}
    for model_file in model_files:
        model_name = os.path.basename(model_file).replace(f"model_{target_group}_", "").replace(".pkl", "")
        try:
            model = joblib.load(model_file)
            models[model_name] = model
        except Exception as e:
            print(f"  ⚠️ Could not load {model_name}: {str(e)}")
    
    return {
        'results': results_df,
        'models': models,
        'model_files': model_files
    }

# Load checkpoint data
checkpoint_low = load_latest_checkpoint(CONFIG['models']['checkpoint_dir'], 'low')
checkpoint_mid = load_latest_checkpoint(CONFIG['models']['checkpoint_dir'], 'mid')

if checkpoint_low is None or checkpoint_mid is None:
    print("❌ Please run '01_model_training.ipynb' first!")
    raise SystemExit






📂 Loading checkpoints for LOW income...
  ✅ Found model: model_low_CatBoost_20260812_034843.pkl
  ✅ Found results: results_low_20260812_034843.csv

📂 Loading checkpoints for MID income...
  ✅ Found model: model_mid_CatBoost_20260812_034843.pkl
  ✅ Found results: results_mid_20260812_034843.csv


In [9]:
# Load Training Data
def load_training_data(config):
    
    try:
        X_train = pd.read_csv(config['data']['train_path'])
        X_val = pd.read_csv(config['data']['val_path'])
        y_train_low = pd.read_csv(config['data']['y_train_low']).iloc[:, 0].values
        y_train_mid = pd.read_csv(config['data']['y_train_mid']).iloc[:, 0].values
        y_val_low = pd.read_csv(config['data']['y_val_low']).iloc[:, 0].values
        y_val_mid = pd.read_csv(config['data']['y_val_mid']).iloc[:, 0].values
        
        print("✅ Training data loaded successfully!")
        print(f"  X_train: {X_train.shape}")
        print(f"  X_val: {X_val.shape}")
        
        return {
            'X_train': X_train,
            'X_val': X_val,
            'y_train_low': y_train_low,
            'y_train_mid': y_train_mid,
            'y_val_low': y_val_low,
            'y_val_mid': y_val_mid
        }
    except Exception as e:
        print(f"❌ Error loading data: {str(e)}")
        return None

data = load_training_data(CONFIG)

if data is None:
    print("❌ Failed to load data. Please check file paths.")
    raise SystemExit

✅ Training data loaded successfully!
  X_train: (144, 39)
  X_val: (48, 39)


In [13]:
# Select Top Models


def select_top_models(checkpoint_data, n_top=3):
    
    if checkpoint_data is None:
        return []
    
    results_df = checkpoint_data['results']
    models_dict = checkpoint_data['models']
    
    # Sort by validation RMSE (ascending)
    results_sorted = results_df.sort_values('RMSE_Val').head(n_top)
    
    top_models = []
    for _, row in results_sorted.iterrows():
        model_name = row['Model']
        if model_name in models_dict:
            top_models.append((model_name, models_dict[model_name]))
            print(f"  ✅ {model_name}: RMSE_Val = {row['RMSE_Val']:.4f}, R2_Val = {row['R2_Val']:.4f}")
        else:
            print(f"  ⚠️ {model_name}: Model object not found")
    
    return top_models

print("\n🏆 TOP 3 MODELS - LOW INCOME:")
top_low = select_top_models(checkpoint_low, n_top=3)

print("\n🏆 TOP 3 MODELS - MIDDLE INCOME:")
top_mid = select_top_models(checkpoint_mid, n_top=3)

# Store selected models for tuning
SELECTED_MODELS = {
    'low': top_low,
    'mid': top_mid
}


🏆 TOP 3 MODELS - LOW INCOME:
  ⚠️ Linear_Regression: Model object not found
  ⚠️ Ridge: Model object not found
  ⚠️ CatBoost: Model object not found

🏆 TOP 3 MODELS - MIDDLE INCOME:
  ⚠️ Linear_Regression: Model object not found
  ⚠️ Ridge: Model object not found
  ⚠️ XGBoost: Model object not found


In [14]:
# Define Optuna search spaces for each model type
def get_search_space(model_name, trial):
    
    if model_name == 'Random_Forest':
        return {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300, step=50),
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
            'max_features': trial.suggest_float('max_features', 0.3, 1.0),
            'bootstrap': trial.suggest_categorical('bootstrap', [True, False])
        }
    
    elif model_name == 'XGBoost':
        return {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300, step=50),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.001, 10, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.001, 10, log=True)
        }
    
    elif model_name == 'LightGBM':
        return {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300, step=50),
            'max_depth': trial.suggest_int('max_depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 10, 100),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 30),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.001, 10, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.001, 10, log=True)
        }
    
    elif model_name == 'CatBoost':
        return {
            'iterations': trial.suggest_int('iterations', 50, 300, step=50),
            'depth': trial.suggest_int('depth', 3, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 10, log=True),
            'border_count': trial.suggest_int('border_count', 32, 255),
            'random_strength': trial.suggest_float('random_strength', 0.1, 10, log=True)
        }
    
    elif model_name in ['Ridge', 'Lasso']:
        return {
            'alpha': trial.suggest_float('alpha', 0.0001, 100, log=True)
        }
    
    else:
        return {}

def create_model_instance(model_name, params, random_state=42):
    
    if model_name == 'Random_Forest':
        return RandomForestRegressor(**params, random_state=random_state, n_jobs=-1)
    elif model_name == 'XGBoost':
        return xgb.XGBRegressor(**params, random_state=random_state, n_jobs=-1)
    elif model_name == 'LightGBM':
        return lgb.LGBMRegressor(**params, random_state=random_state, n_jobs=-1, verbose=-1)
    elif model_name == 'CatBoost':
        return CatBoostRegressor(**params, random_seed=random_state, verbose=False, thread_count=-1)
    elif model_name == 'Ridge':
        return Ridge(**params, random_state=random_state)
    elif model_name == 'Lasso':
        return Lasso(**params, random_state=random_state)
    else:
        raise ValueError(f"Unknown model: {model_name}")

print("✅ Search spaces defined for all models!")

✅ Search spaces defined for all models!


In [15]:
# Define the objective function for Optuna optimization
def objective(trial, model_name, X_train, y_train, X_val, y_val, cv_folds=5):
    
    try:
        # Get hyperparameters for this trial
        params = get_search_space(model_name, trial)
        
        # Create model instance
        model = create_model_instance(model_name, params)
        
        # Use TimeSeriesSplit for cross-validation
        tscv = TimeSeriesSplit(n_splits=cv_folds)
        cv_scores = []
        
        # Combine train and val for CV
        X_combined = np.vstack([X_train, X_val])
        y_combined = np.concatenate([y_train, y_val])
        
        for train_idx, val_idx in tscv.split(X_combined):
            X_cv_train, X_cv_val = X_combined[train_idx], X_combined[val_idx]
            y_cv_train, y_cv_val = y_combined[train_idx], y_combined[val_idx]
            
            model.fit(X_cv_train, y_cv_train)
            y_pred = model.predict(X_cv_val)
            rmse = np.sqrt(mean_squared_error(y_cv_val, y_pred))
            cv_scores.append(rmse)
        
        # Return mean CV RMSE (negative for minimization)
        mean_rmse = np.mean(cv_scores)
        
        # Prune unpromising trials
        if trial.should_prune():
            raise optuna.TrialPruned()
        
        return -mean_rmse  # Negative because Optuna minimizes
        
    except Exception as e:
        # Return a very poor score for failed trials
        print(f"  ⚠️ Trial failed: {str(e)[:100]}")
        return 1e6  # Large positive value = poor performance

In [16]:
# Run Optuna optimization for each selected model and income group
def run_optimization(model_name, model, X_train, y_train, X_val, y_val, 
                     target_group, n_trials=100, cv_folds=5):
    
    print(f"\n{'='*60}")
    print(f"🔬 Optimizing {model_name} - {target_group.upper()} Income")
    print(f"{'='*60}")
    
    # Create study
    study_name = f"{model_name}_{target_group}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    storage_name = f"sqlite:///{CONFIG['models']['mlflow_dir']}/{study_name}.db"
    
    study = optuna.create_study(
        study_name=study_name,
        storage=storage_name,
        sampler=TPESampler(seed=CONFIG['tuning']['random_state']),
        pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=10),
        direction='minimize',
        load_if_exists=True
    )
    
    # Run optimization
    try:
        study.optimize(
            lambda trial: -objective(
                trial, model_name, X_train, y_train, X_val, y_val, cv_folds
            ),
            n_trials=n_trials,
            timeout=CONFIG['tuning']['timeout'],
            show_progress_bar=True
        )
        
        print(f"\n✅ Optimization complete!")
        print(f"  Best RMSE: {-study.best_value:.4f}")
        print(f"  Best params: {study.best_params}")
        
        return study
        
    except Exception as e:
        print(f"❌ Optimization failed: {str(e)}")
        return None

In [17]:
# Run optimization for all selected models
# Convert data to numpy arrays for faster processing
X_train_np = data['X_train'].values
X_val_np = data['X_val'].values

# Store all studies
studies = {
    'low': {},
    'mid': {}
}

# Tune low income models
print("\n" + "🔴" * 30)
print("TUNING LOW INCOME MODELS")
print("🔴" * 30)

for model_name, model in SELECTED_MODELS['low']:
    study = run_optimization(
        model_name=model_name,
        model=model,
        X_train=X_train_np,
        y_train=data['y_train_low'],
        X_val=X_val_np,
        y_val=data['y_val_low'],
        target_group='low',
        n_trials=CONFIG['tuning']['n_trials'],
        cv_folds=CONFIG['tuning']['cv_folds']
    )
    if study:
        studies['low'][model_name] = study

# Tune middle income models
print("\n" + "🟡" * 30)
print("TUNING MIDDLE INCOME MODELS")
print("🟡" * 30)

for model_name, model in SELECTED_MODELS['mid']:
    study = run_optimization(
        model_name=model_name,
        model=model,
        X_train=X_train_np,
        y_train=data['y_train_mid'],
        X_val=X_val_np,
        y_val=data['y_val_mid'],
        target_group='mid',
        n_trials=CONFIG['tuning']['n_trials'],
        cv_folds=CONFIG['tuning']['cv_folds']
    )
    if study:
        studies['mid'][model_name] = study

print("\n" + "="*60)
print("✅ ALL OPTIMIZATION COMPLETE!")
print("="*60)


🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴
TUNING LOW INCOME MODELS
🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴🔴

🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡
TUNING MIDDLE INCOME MODELS
🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡🟡

✅ ALL OPTIMIZATION COMPLETE!


In [18]:
# Visualization
def plot_optimization_results(study, model_name, target_group):
    
    if study is None:
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot optimization history
    ax1 = axes[0]
    history_df = study.trials_dataframe()
    history_df['value'] = -history_df['value']  # Convert back to RMSE
    ax1.plot(history_df['number'], history_df['value'], 'b-', alpha=0.5)
    ax1.set_xlabel('Trial')
    ax1.set_ylabel('RMSE')
    ax1.set_title(f'{model_name} - {target_group.upper()} Income\nOptimization History')
    ax1.grid(True, alpha=0.3)
    
    # Plot parameter importance
    ax2 = axes[1]
    importance = optuna.importance.get_param_importances(study)
    params = list(importance.keys())
    values = list(importance.values())
    
    # Sort by importance
    sorted_idx = np.argsort(values)[::-1]
    params_sorted = [params[i] for i in sorted_idx]
    values_sorted = [values[i] for i in sorted_idx]
    
    # Take top 10
    params_sorted = params_sorted[:10]
    values_sorted = values_sorted[:10]
    
    ax2.barh(params_sorted, values_sorted)
    ax2.set_xlabel('Importance')
    ax2.set_title('Parameter Importance')
    ax2.invert_yaxis()
    
    plt.tight_layout()
    
    # Save figure
    os.makedirs('../reports/figures/tuning/', exist_ok=True)
    plt.savefig(f'../reports/figures/tuning/optuna_{model_name}_{target_group}.png', 
                dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Figure saved: optuna_{model_name}_{target_group}.png")

# Plot results for all studies
for target_group, model_studies in studies.items():
    for model_name, study in model_studies.items():
        plot_optimization_results(study, model_name, target_group)

In [19]:
# Save the best models from optimization
def save_optimized_models(studies, X_train, y_train):
    
    print("\n💾 Saving optimized models...")
    
    for target_group, model_studies in studies.items():
        print(f"\n  📁 {target_group.upper()} Income:")
        
        for model_name, study in model_studies.items():
            if study is None:
                continue
            
            # Get best parameters
            best_params = study.best_params
            
            # Create model with best parameters
            try:
                model = create_model_instance(
                    model_name, 
                    best_params,
                    random_state=CONFIG['tuning']['random_state']
                )
                
                # Train on full training data
                model.fit(X_train, y_train)
                
                # Save model
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                model_path = f"{CONFIG['models']['tuned_dir']}/tuned_{model_name}_{target_group}_{timestamp}.pkl"
                joblib.dump(model, model_path)
                
                # Save parameters
                params_path = f"{CONFIG['models']['tuned_dir']}/params_{model_name}_{target_group}_{timestamp}.json"
                with open(params_path, 'w') as f:
                    json.dump(best_params, f, indent=2)
                
                print(f"    ✅ {model_name}: {model_path}")
                
            except Exception as e:
                print(f"    ❌ {model_name}: Failed to save - {str(e)}")

# Prepare data
X_train_full = data['X_train'].values
y_train_low = data['y_train_low']
y_train_mid = data['y_train_mid']

# Save low income models
save_optimized_models(
    {'low': studies['low']}, 
    X_train_full, 
    y_train_low
)

# Save middle income models
save_optimized_models(
    {'mid': studies['mid']}, 
    X_train_full, 
    y_train_mid
)

print("\n✅ All optimized models saved!")


💾 Saving optimized models...

  📁 LOW Income:

💾 Saving optimized models...

  📁 MID Income:

✅ All optimized models saved!


In [20]:
# Log all experiments to MLflow
def log_to_mlflow(studies, target_group):
    
    mlflow.set_tracking_uri(f"file:{CONFIG['models']['mlflow_dir']}")
    
    for model_name, study in studies.items():
        if study is None:
            continue
        
        experiment_name = f"{model_name}_{target_group}_tuning"
        mlflow.set_experiment(experiment_name)
        
        with mlflow.start_run(run_name=f"tuning_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
            # Log best parameters
            mlflow.log_params(study.best_params)
            
            # Log best value
            mlflow.log_metric("best_rmse", -study.best_value)
            
            # Log study details
            mlflow.log_param("n_trials", len(study.trials))
            mlflow.log_param("n_failed_trials", len([t for t in study.trials if t.state == optuna.trial.TrialState.FAILED]))
            
            # Log best model
            model = create_model_instance(model_name, study.best_params)
            mlflow.sklearn.log_model(model, "best_model")
            
            print(f"  ✅ Logged {model_name} ({target_group}) to MLflow")

# Log all experiments
print("\n📊 Logging to MLflow...")

log_to_mlflow(studies['low'], 'low')
log_to_mlflow(studies['mid'], 'mid')

print("\n✅ All experiments logged to MLflow!")
print(f"📁 MLflow directory: {CONFIG['models']['mlflow_dir']}")


📊 Logging to MLflow...

✅ All experiments logged to MLflow!
📁 MLflow directory: ../models/mlflow/


In [21]:
# Summary Report
print("\n" + "="*60)
print("📋 PHASE 2 COMPLETE: HYPERPARAMETER TUNING")
print("="*60)

print("\n🏆 OPTIMIZED MODEL PERFORMANCE:")

for target_group, model_studies in studies.items():
    print(f"\n  📊 {target_group.upper()} Income:")
    for model_name, study in model_studies.items():
        if study:
            best_rmse = -study.best_value
            best_params = study.best_params
            n_trials = len(study.trials)
            
            print(f"\n    🔹 {model_name}:")
            print(f"       Best RMSE: {best_rmse:.4f}")
            print(f"       Trials: {n_trials}")
            print(f"       Top Parameters:")
            # Show top 5 most important parameters
            importance = optuna.importance.get_param_importances(study)
            top_params = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:5]
            for param, imp in top_params:
                value = best_params.get(param, 'N/A')
                print(f"          - {param}: {value} (importance: {imp:.3f})")

print("\n📂 OUTPUTS GENERATED:")
print(f"  ✅ Tuned models: {CONFIG['models']['tuned_dir']}")
print(f"  ✅ MLflow logs: {CONFIG['models']['mlflow_dir']}")
print(f"  ✅ Optimization plots: ../reports/figures/tuning/")


📋 PHASE 2 COMPLETE: HYPERPARAMETER TUNING

🏆 OPTIMIZED MODEL PERFORMANCE:

  📊 LOW Income:

  📊 MID Income:

📂 OUTPUTS GENERATED:
  ✅ Tuned models: ../models/tuned/
  ✅ MLflow logs: ../models/mlflow/
  ✅ Optimization plots: ../reports/figures/tuning/
